# Optimiser Comparison — Binary Classification on the Moons Dataset

Train the **same MLP classifier** with SGD, Momentum, and Adam; compare how their distinct personalities (inertia, adaptation, look-ahead) affect convergence.

Each optimiser starts from **identical** initial weights, trains for the same number of epochs
on the same mini-batches, and is evaluated on the same validation split.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.autograd import Value
from core.nn import (
    SGD,
    Adam,
    BCELoss,
    DataLoader,
    Linear,
    Momentum,
    ReLU,
    Sequential,
    Sigmoid,
)

In [ ]:
# =====================================================================
# 1. Generate the Moons Dataset (from scratch, no sklearn)
# =====================================================================

np.random.seed(42)

n_samples = 500
noise = 0.20
n_per_class = n_samples // 2

theta = np.linspace(0, np.pi, n_per_class)
x0 = np.cos(theta) + np.random.randn(n_per_class) * noise
y0 = np.sin(theta) + np.random.randn(n_per_class) * noise
x1 = 1 - np.cos(theta) + np.random.randn(n_per_class) * noise
y1 = 0.5 - np.sin(theta) + np.random.randn(n_per_class) * noise

X = np.vstack([np.column_stack([x0, y0]), np.column_stack([x1, y1])])
y = np.vstack([np.zeros((n_per_class, 1)), np.ones((n_per_class, 1))])

indices = np.random.permutation(n_samples)
X, y = X[indices], y[indices]

# Train / val split (80 / 20)
split = int(0.8 * n_samples)
x_train, x_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Train: {x_train.shape}, Val: {x_val.shape}")
print(f"Class balance (train): {y_train.mean():.2f}")

# Visualise
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(
    x_train[y_train.ravel() == 0, 0],
    x_train[y_train.ravel() == 0, 1],
    s=10,
    alpha=0.7,
    label="class 0",
)
ax.scatter(
    x_train[y_train.ravel() == 1, 0],
    x_train[y_train.ravel() == 1, 1],
    s=10,
    alpha=0.7,
    label="class 1",
)
ax.legend()
ax.set_title("Moons Dataset")
ax.set_aspect("equal")
plt.show()

In [ ]:
# =====================================================================
# 2. Model factory — identical architecture & init for every run
# =====================================================================


def make_model(seed: int = 42):
    """Create a fresh classifier with the same architecture and init."""
    np.random.seed(seed)
    model = Sequential(
        [
            Linear(2, 16),
            ReLU(),
            Linear(16, 16),
            ReLU(),
            Linear(16, 1),
            Sigmoid(),
        ]
    )
    return model


def copy_state(model):
    """Snapshot the current model parameters so we can clone it."""
    return {id(p): (p.data.copy(), p.grad) for p in model.parameters()}


def restore_state(model, state):
    """Restore a previously snapshotted state into a model."""
    for p in model.parameters():
        data, grad = state[id(p)]
        p.data = data.copy()
        p.grad = grad


# Verify that two models with the same seed are identical
m1 = make_model(42)
m2 = make_model(42)
w1 = [p.data.copy() for p in m1.parameters()]
w2 = [p.data.copy() for p in m2.parameters()]
all_close = all(np.allclose(a, b) for a, b in zip(w1, w2))
print(f"Seed reproducibility check: {'OK (identical init)' if all_close else 'FAILED'}")

In [ ]:
# =====================================================================
# 3. Training harness — run one optimiser and record history
# =====================================================================

loss_fn = BCELoss()
BATCH_SIZE = 16
EPOCHS = 300


def train_optimiser(opt_class, opt_kwargs, model, label=""):
    """Train `model` with the given optimiser class and kwargs.

    Uses the module-level train_loader and validation data.
    Returns dict of history arrays.
    """
    loader = DataLoader(x_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
    opt = opt_class(model.parameters(), **opt_kwargs)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for bx_np, by_np in loader:
            bx, by = Value(bx_np), Value(by_np)
            pred = model(bx)
            loss = loss_fn(pred, by)

            opt.zero_grad()
            loss.backward()
            opt.step()

            epoch_loss += loss.data

        history["train_loss"].append(epoch_loss / len(loader))

        # Validation (no grad needed)
        vx, vy = Value(x_val), Value(y_val)
        vpred = model(vx)
        vloss = loss_fn(vpred, vy).data
        vacc = ((vpred.data > 0.5).astype(float) == y_val).mean()
        history["val_loss"].append(vloss)
        history["val_acc"].append(vacc)

    # Convert to numpy arrays
    for k, v in history.items():
        history[k] = np.array(v)

    return history


# Config: per-optimiser hyper-parameters (tuned for this dataset)
OPT_CONFIGS = [
    ("SGD", SGD, {"lr": 0.01}),
    ("Momentum", Momentum, {"lr": 0.003, "momentum": 0.9}),
    ("Adam", Adam, {"lr": 0.001, "betas": (0.9, 0.999)}),
]

print(f"Training {len(OPT_CONFIGS)} optimisers for {EPOCHS} epochs each...\n")

all_history = {}
all_models = {}

for label, opt_class, kwargs in OPT_CONFIGS:
    # Start from the same initial weights every time
    model = make_model(seed=42)
    hist = train_optimiser(opt_class, kwargs, model, label=label)
    all_history[label] = hist
    all_models[label] = model

    final_acc = hist["val_acc"][-1]
    final_loss = hist["val_loss"][-1]
    print(f"  {label:>10}: val loss {final_loss:.4f}  |  val acc {final_acc:.3f}")

In [ ]:
# =====================================================================
# 4. Loss & accuracy convergence curves
# =====================================================================

COLOURS = {"SGD": "#e41a1c", "Momentum": "#377eb8", "Adam": "#4daf4a"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# --- Loss curves ---
for label, _, _ in OPT_CONFIGS:
    h = all_history[label]
    ax1.plot(
        h["train_loss"],
        color=COLOURS[label],
        linestyle="--",
        alpha=0.5,
        label=f"{label} (train)",
    )
    ax1.plot(
        h["val_loss"],
        color=COLOURS[label],
        linestyle="-",
        linewidth=1.5,
        label=f"{label} (val)",
    )

ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Loss Convergence")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Accuracy curves ---
for label, _, _ in OPT_CONFIGS:
    h = all_history[label]
    ax2.plot(h["val_acc"], color=COLOURS[label], linewidth=1.5, label=label)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy")
ax2.legend(fontsize=8)
ax2.set_ylim(0.5, 1.0)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# 5. Decision boundaries — side by side
# =====================================================================

x_min, x_max = -1.5, 2.5
y_min, y_max = -1.0, 2.0
h = 0.02
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, _, _) in zip(axes, OPT_CONFIGS):
    model = all_models[label]
    model.eval()

    Z = model(Value(grid)).data.reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=["#ffcccc", "#cceeff"], alpha=0.8)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.2)

    ax.scatter(
        x_train[y_train.ravel() == 0, 0],
        x_train[y_train.ravel() == 0, 1],
        s=6,
        alpha=0.5,
        label="class 0",
    )
    ax.scatter(
        x_train[y_train.ravel() == 1, 0],
        x_train[y_train.ravel() == 1, 1],
        s=6,
        alpha=0.5,
        label="class 1",
    )

    acc = all_history[label]["val_acc"][-1]
    loss = all_history[label]["val_loss"][-1]
    ax.set_title(f"{label}\nval loss={loss:.4f}  acc={acc:.3f}")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect("equal")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# 6. Convergence speed — epochs to reach 90% / 95% / 97% val accuracy
# =====================================================================

print(
    f"{'Optimiser':>10} | {'→ 90%':>8} | {'→ 95%':>8} | {'→ 97%':>8} | {'Final acc':>8} | {'Final loss':>8}"
)
print("-" * 60)

for label, _, _ in OPT_CONFIGS:
    accs = all_history[label]["val_acc"]
    losses = all_history[label]["val_loss"]

    def first_above(threshold):
        idx = np.where(accs >= threshold)[0]
        return f"{idx[0] + 1:4d}" if len(idx) else "  —"

    print(
        f"{label:>10} | {first_above(0.90):>8} | {first_above(0.95):>8} | {first_above(0.97):>8}"
        f" | {accs[-1]:>7.3f}  | {losses[-1]:>7.4f}"
    )

### Discussion

On this moons classification task:

- SGD (lr=0.01) is stable but slow — it takes ~100+ epochs to reach 95%
  because vanilla gradient descent has no acceleration.
- Momentum (lr=0.003, β=0.9) converges faster once velocity builds up.
  The lower LR is needed because momentum amplifies the effective step.
- Adam (lr=0.001) is the fastest early on — bias-corrected moments
  give it a robust per-parameter adaptive step from the first epoch.

All three hit ~97% validation accuracy eventually — the model is
expressive enough (2 hidden layers, 16 units) to separate the moons.
The difference is in *how quickly* they get there, which matters when
training large models where every epoch costs.

This behaviour matches the toy-surface demo from the previous notebook:
Adam adapts fastest early, Momentum catches up, and SGD plods along
but converges reliably with a well-tuned LR.